# Normalisation de GSE149689

Construction d'un objet log-normalisé tout en conservant les counts UMI bruts et la sélection finale de 2 000 HVG.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from IPython.display import display

## Chargement et contrôles initiaux

In [2]:
adata = sc.read_h5ad("../data/processed/GSE149689_singlets.h5ad")

x_is_sparse = sparse.issparse(adata.X)
if not x_is_sparse:
    raise AssertionError("adata.X doit être sparse.")
x_values = adata.X.data
initial_checks = {
    "dimensions_58825_x_33538": adata.shape == (58825, 33538),
    "20_sample_id": "sample_id" in adata.obs and adata.obs["sample_id"].nunique() == 20,
    "X_sparse": x_is_sparse,
    "counts_nonnegative": bool(np.all(x_values >= 0)),
    "counts_integer_valued": bool(np.all(np.equal(x_values, np.floor(x_values)))),
    "no_log1p_metadata": "log1p" not in adata.uns,
}
print(adata)
display(pd.Series(initial_checks, name="passed").to_frame())
if not all(initial_checks.values()):
    raise AssertionError(f"Échec des contrôles initiaux : {initial_checks}")

AnnData object with n_obs × n_vars = 58825 × 33538
    obs: 'sample_id', 'subject_id', 'disease', 'severity', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet'
    var: 'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'


,passed
dimensions_58825_x_33538,True
20_sample_id,True
X_sparse,True
counts_nonnegative,True
counts_integer_valued,True
no_log1p_metadata,True


## Préservation des counts et sélection finale des HVG

In [3]:
adata.layers["counts"] = adata.X.copy()

def sparse_fingerprint(matrix):
    if not sparse.issparse(matrix):
        raise TypeError("Une matrice sparse est requise pour l'empreinte.")
    matrix = matrix.tocsr(copy=False)
    digest = hashlib.sha256()
    digest.update(np.asarray(matrix.shape, dtype=np.int64).tobytes())
    digest.update(matrix.indptr.tobytes())
    digest.update(matrix.indices.tobytes())
    digest.update(matrix.data.tobytes())
    return digest.hexdigest()

initial_counts_fingerprint = sparse_fingerprint(adata.layers["counts"])

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="seurat_v3_paper",
    batch_key="sample_id",
    layer="counts",
    inplace=True,
)
n_hvg = int(adata.var["highly_variable"].sum())
print("Nombre de HVG :", n_hvg)
if n_hvg != 2000:
    raise AssertionError(f"2 000 HVG attendus, {n_hvg} obtenus.")

Nombre de HVG : 2000


## Distribution des sommes de counts bruts par cellule

In [4]:
raw_totals = np.asarray(adata.X.sum(axis=1)).ravel()
raw_total_summary = pd.Series({
    "min": raw_totals.min(),
    "Q1": np.quantile(raw_totals, 0.25),
    "median": np.median(raw_totals),
    "mean": raw_totals.mean(),
    "Q3": np.quantile(raw_totals, 0.75),
    "max": raw_totals.max(),
}, name="raw_total_counts")
display(raw_total_summary.to_frame())

,raw_total_counts
min,230.000000
Q1,3941.000000
median,5563.000000
mean,6852.731934
Q3,8397.000000
max,152409.000000


## Normalisation à une somme cible de 10 000

In [5]:
example_cell_positions = np.arange(5)
example_gene_positions = np.argsort(
    np.asarray(adata.X[example_cell_positions].sum(axis=0)).ravel()
)[-5:][::-1]
example_cells = adata.obs_names[example_cell_positions].tolist()
example_genes = adata.var_names[example_gene_positions].tolist()
example_raw_values = adata.layers["counts"][example_cell_positions][:, example_gene_positions].toarray()

sc.pp.normalize_total(adata, target_sum=1e4)

normalized_totals = np.asarray(adata.X.sum(axis=1)).ravel()
normalization_checks = {
    "X_remains_sparse": sparse.issparse(adata.X),
    "dimensions_unchanged": adata.shape == (58825, 33538),
    "cell_sums_approximately_10000": bool(np.allclose(normalized_totals, 1e4, rtol=1e-5, atol=0.1)),
}
display(pd.Series(normalization_checks, name="passed").to_frame())
if not all(normalization_checks.values()):
    raise AssertionError(f"Échec des contrôles après normalize_total : {normalization_checks}")

totals_examples = pd.DataFrame({
    "barcode": example_cells,
    "total_raw_counts": raw_totals[example_cell_positions],
    "total_after_normalize_total": normalized_totals[example_cell_positions],
})
display(totals_examples)

example_normalized_values = adata.X[example_cell_positions][:, example_gene_positions].toarray()

,passed
X_remains_sparse,True
dimensions_unchanged,True
cell_sums_approximately_10000,True


,barcode,total_raw_counts,total_after_normalize_total
0,AAACCCAAGGGCAATC-1,15679.0,10000.0
1,AAACCCACAGCTGAAG-1,3630.0,10000.0
2,AAACCCAGTCTTCGAA-1,6796.0,10000.0
3,AAACCCAGTTCCGCTT-1,6803.0,10000.0
4,AAACGAAAGGGAGGTG-1,819.0,10000.0


## Transformation log1p et contrôles

In [6]:
sc.pp.log1p(adata)

counts_layer_values = adata.layers["counts"].data
post_log_checks = {
    "log1p_metadata_present": "log1p" in adata.uns,
    "X_sparse": sparse.issparse(adata.X),
    "dimensions_unchanged": adata.shape == (58825, 33538),
    "X_nonnegative": bool(np.all(adata.X.data >= 0)),
    "counts_layer_sparse": sparse.issparse(adata.layers["counts"]),
    "counts_layer_nonnegative": bool(np.all(counts_layer_values >= 0)),
    "counts_layer_integer_valued": bool(np.all(np.equal(counts_layer_values, np.floor(counts_layer_values)))),
    "counts_layer_unchanged": sparse_fingerprint(adata.layers["counts"]) == initial_counts_fingerprint,
    "exactly_2000_HVG": int(adata.var["highly_variable"].sum()) == 2000,
}
display(pd.Series(post_log_checks, name="passed").to_frame())
if not all(post_log_checks.values()):
    raise AssertionError(f"Échec des contrôles après log1p : {post_log_checks}")

,passed
log1p_metadata_present,True
X_sparse,True
dimensions_unchanged,True
X_nonnegative,True
counts_layer_sparse,True
counts_layer_nonnegative,True
counts_layer_integer_valued,True
counts_layer_unchanged,True
exactly_2000_HVG,True


## Exemple de trois représentations

In [7]:
example_log1p_values = adata.X[example_cell_positions][:, example_gene_positions].toarray()
representation_rows = []
for cell_idx, barcode in enumerate(example_cells):
    for gene_idx, gene_name in enumerate(example_genes):
        representation_rows.append({
            "barcode": barcode,
            "gene_name": gene_name,
            "raw_count": example_raw_values[cell_idx, gene_idx],
            "normalized_before_log1p": example_normalized_values[cell_idx, gene_idx],
            "log1p_value": example_log1p_values[cell_idx, gene_idx],
        })
representation_example = pd.DataFrame(representation_rows)
display(representation_example)

,barcode,gene_name,raw_count,normalized_before_log1p,log1p_value
0,AAACCCAAGGGCAATC-1,MALAT1,766.0,488.551575,6.193490
1,AAACCCAAGGGCAATC-1,MT-CO3,197.0,125.645775,4.841394
2,AAACCCAAGGGCAATC-1,EEF1A1,176.0,112.252060,4.729616
3,AAACCCAAGGGCAATC-1,RPL10,190.0,121.181206,4.805505
4,AAACCCAAGGGCAATC-1,MT-CO2,186.0,118.630020,4.784404
5,AAACCCACAGCTGAAG-1,MALAT1,283.0,779.614319,6.660081
6,AAACCCACAGCTGAAG-1,MT-CO3,111.0,305.785126,5.726148
7,AAACCCACAGCTGAAG-1,EEF1A1,44.0,121.212120,4.805758
8,AAACCCACAGCTGAAG-1,RPL10,38.0,104.683197,4.660446
9,AAACCCACAGCTGAAG-1,MT-CO2,75.0,206.611572,5.335669


## Interprétation

normalize_total(target_sum=1e4) place les cellules sur une échelle comparable, mais ne signifie pas qu'elles possédaient réellement 10 000 UMI. La transformation log1p compresse les grandes valeurs. adata.X contient maintenant l'expression log-normalisée, tandis que adata.layers["counts"] conserve les UMI bruts. Les 2 000 HVG sont uniquement marqués dans adata.var et les 33 538 gènes restent présents.

## Sauvegarde

In [8]:
output_path = Path("../data/processed/GSE149689_log_normalized.h5ad")
adata.write_h5ad(output_path)
print("Objet sauvegardé :", output_path)
print("Dimensions :", adata.shape)
print("HVG marqués :", int(adata.var["highly_variable"].sum()))
print("Taille du fichier :", output_path.stat().st_size, "octets")

Objet sauvegardé : ../data/processed/GSE149689_log_normalized.h5ad
Dimensions : (58825, 33538)
HVG marqués : 2000
Taille du fichier : 1812253127 octets
